In [1]:
# Labial 

import sys

import numpy as np 
import pandas as pd 
import os 
import pickle
from pathlib import Path
REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT / "code"))
from cell_groups import *
from color_utils import * 
from make_network import *
from make_network_utils import *
from analysis_of_simulation_result import * 



In [2]:
# # Rank data to Set color of nodes 
# def make_rank_color_info(rank_df, top_n=20, base_color="#FF0066", default_color="#777777"):
#     num_top_ranked = (rank_df <= top_n).sum(axis=1)

#     return {
#         tid: (
#             get_graded_color(base_color, int(num))
#             if num > 2
#             else default_color
#         )
#         for tid, num in num_top_ranked.items()
#     }


Rank_DATA_DIR = Path.cwd().resolve().parents[0]/'figure5Q,R/data'
tarsal_sweet_rank_df = pd.read_parquet(Rank_DATA_DIR/'tarsal_sweet_rank.parquet')
labial_sweet_rank_df = pd.read_parquet(Rank_DATA_DIR/'labial_sweet_rank.parquet')
tarsal_geosmin_rank_df = pd.read_parquet(Rank_DATA_DIR/'tarsal_geosmin_rank.parquet')
labial_geosmin_rank_df = pd.read_parquet(Rank_DATA_DIR/'labial_geosmin_rank.parquet')


# color_info_tarsal_sweet = make_rank_color_info(
#     tarsal_sweet_rank_df,
#     top_n=20,
#     base_color="#0066FF",
# )

# color_info_labial_sweet = make_rank_color_info(
#     labial_sweet_rank_df,
#     top_n=20,
#     base_color="#00AA66",
# )

# color_info_tarsal_geosmin = make_rank_color_info(
#     tarsal_geosmin_rank_df,
#     top_n=20,
#     base_color="#FF0066",
# )


# color_info_labial_geosmin = make_rank_color_info(
#     labial_geosmin_rank_df,
#     top_n=20,
#     base_color="#FF0066",
# )

In [3]:
sez_ids = np.concatenate(list(sez.values()))
fid2sez_g = {c:t for t,cids in sez.items() for c in cids}

gids = np.unique(np.concatenate([tarsal_sweet_rank_df.index,labial_sweet_rank_df.index,tarsal_geosmin_rank_df.index,labial_geosmin_rank_df.index]))

canonical_name = []
for g in gids:
    g = int(g)
    fids = target_ids_valid_all[g_info_all==g]
    if np.any(np.isin(fids,sez_ids)):
        canonical_name.append(fid2sez_g[fids[np.isin(fids,sez_ids)][0]])
    else:
        canonical_name.append(str(g))
gid2canonical_name = dict(zip(gids,canonical_name))
gid2canonical_name['control'] = 'control'

In [6]:
BASE_DIR = Path(
    "/volume_4/research/seongbong/flywire/geosmin_project"
)

UPDATED_BASE_DIR = Path(
    "/volume_4/research/seongbong/flywire/"
    "geosmin_project_version_update"
)

graph_paths = {
    "tarsal_sweet": (
        BASE_DIR
        / "figure/figure5_new-order/functional_network/"
          "sweet/tarsal/tarsal_hm_thresholded_0.1.gexf"
    ),
    "labial_sweet": (
        BASE_DIR
        / "figure/figure5_new-order/functional_network/"
          "sweet/labial/labial_hm_thresholded_0.1.gexf"
    ),
    "tarsal_geosmin": (
        BASE_DIR
        / "figure/figure5_new-order/functional_network/"
          "geosmin/tarsal/"
          "tarsal_av1a1_hm_thresholded_0.1_0.2.gexf"
    ),
    "labial_geosmin": (
        BASE_DIR
        / "figure/figure8/"
          "Network_with_necessity_with_spike_pattern_fixed/"
          "Merged_network_new_criterion/labial_av1a1/"
          "labial_av1a1_hm_thresholded_0.1_0.2.gexf"
    ),
}

graphs = load_gexf_graphs(graph_paths)

tarsal_sweetG = graphs["tarsal_sweet"]
labial_sweetG = graphs["labial_sweet"]
tarsal_sgG = graphs["tarsal_geosmin"]
labial_sgG = graphs["labial_geosmin"]

In [17]:
TOP_N = 20
MIN_COUNT = 1
N_CONCENTRATIONS = tarsal_sweet_rank_df.shape[1]

sweet_rank_counts, in_network_sweet = (
    get_rank_counts_and_selected_nodes(
        tarsal_sweet_rank_df,
        top_n=TOP_N,
        min_count=MIN_COUNT,
    )
)

geosmin_rank_counts, in_network_av1a1 = (
    get_rank_counts_and_selected_nodes(
        tarsal_geosmin_rank_df,
        top_n=TOP_N,
        min_count=MIN_COUNT,
    )
)


c_to_add_sweet = group_ids_to_cell_ids(
    in_network_sweet,
    target_ids_valid_all,
    g_info_all,
)

c_to_add_av1a1 = group_ids_to_cell_ids(
    in_network_av1a1,
    target_ids_valid_all,
    g_info_all,
)

color_info = {}

# 다른 modality의 위치 보정용 node
set_gray_node_colors(
    color_info,
    labial_sgG.nodes,
)


sweet_colors = make_rank_color_info(
    sweet_rank_counts,
    gid2canonical_name=gid2canonical_name,
    base_color="#0066FF",
    max_count=N_CONCENTRATIONS,
)

geosmin_colors = make_rank_color_info(
    geosmin_rank_counts,
    gid2canonical_name=gid2canonical_name,
    base_color="#FF0066",
    max_count=N_CONCENTRATIONS,
)

color_info.update(sweet_colors)
color_info.update(geosmin_colors)

set_gray_node_colors(
    color_info,
    [
        "atGRN",
        "TPN1",
        "MN9",
        "av1a1",
        "DA2_PN",
        "Or56a",
    ],
)



cells_in_new_net = combine_unique_cells(
    c_to_add_sweet,
    atGRNs,
    TPN1,
    mn9,
    c_to_add_av1a1,
    av1a1,
    or56a,
    da2pn,
)

thisG = make_meta_network(cells_in_new_net)


dummy_node_path = (
    UPDATED_BASE_DIR
    / "figure5/functional_network_final-base_PER/"
      "geosmin_labial/labial_core_nodes.pkl"
)

thisG, dummy_nodes = add_dummy_nodes_from_pickle(
    thisG,
    pickle_path=dummy_node_path,
    color_info=color_info,
)


thisG = assign_node_modalities(
    thisG,
    tarsal_nodes=tarsal_sweetG.nodes,
    labial_nodes=labial_sweetG.nodes,
    dummy_nodes=dummy_nodes,
)


position_csv_path = (
    UPDATED_BASE_DIR
    / "figure5/functional_network_final-base_PER/"
      "merged_using_first_positions_0_1000.csv"
)

node_locations = load_node_locations(position_csv_path)

node_locations = add_missing_node_locations(
    thisG,
    node_locations,
    reference_node="horntail",
    x_spacing=100,
)

metaG_with_pos = set_node_location(
    thisG,
    node_locations,
)

metaG_with_pos_color = set_node_color(
    metaG_with_pos,
    color_info,
)

core_graph = threshold_and_transform_edges(
    metaG_with_pos_color,
    min_weight=7,
    weight_transform=np.log2,
)

core_graph = set_edge_color(core_graph)

nx.write_gexf(
    core_graph,
    "tarsal_av1a1_union.gexf",
)

core_graph_without_recurrent = remove_cross_modality_edges(
    core_graph,
    source_modality="tarsal",
    target_modality="geosmin",
)

core_graph_without_recurrent = set_edge_color(
    core_graph_without_recurrent
)

nx.write_gexf(
    core_graph_without_recurrent,
    "tarsal_av1a1_union_remove_recurrent.gexf",
)

In [8]:
TOP_N = 20
MIN_COUNT = 1
N_CONCENTRATIONS = labial_sweet_rank_df.shape[1]

sweet_rank_counts, in_network_sweet = (
    get_rank_counts_and_selected_nodes(
        labial_sweet_rank_df,
        top_n=TOP_N,
        min_count=MIN_COUNT,
    )
)

geosmin_rank_counts, in_network_av1a1 = (
    get_rank_counts_and_selected_nodes(
        labial_geosmin_rank_df,
        top_n=TOP_N,
        min_count=MIN_COUNT,
    )
)


c_to_add_sweet = group_ids_to_cell_ids(
    in_network_sweet,
    target_ids_valid_all,
    g_info_all,
)

c_to_add_av1a1 = group_ids_to_cell_ids(
    in_network_av1a1,
    target_ids_valid_all,
    g_info_all,
)


color_info = {}

# 반대 modality의 geosmin network node를 위치 보정용 회색 node로 사용
set_gray_node_colors(
    color_info,
    tarsal_sgG.nodes,
)


sweet_colors = make_rank_color_info(
    sweet_rank_counts,
    gid2canonical_name=gid2canonical_name,
    base_color="#00AA66",
    max_count=N_CONCENTRATIONS,
)

geosmin_colors = make_rank_color_info(
    geosmin_rank_counts,
    gid2canonical_name=gid2canonical_name,
    base_color="#FF0066",
    max_count=N_CONCENTRATIONS,
)

color_info.update(sweet_colors)
color_info.update(geosmin_colors)


set_gray_node_colors(
    color_info,
    [
        "MN9",
        "av1a1",
        "DA2_PN",
        "Or56a",
    ],
)


cells_in_new_net = combine_unique_cells(
    c_to_add_sweet,
    labial_sweet,
    mn9,
    c_to_add_av1a1,
    av1a1,
    or56a,
    da2pn,
)

thisG = make_meta_network(cells_in_new_net)


dummy_node_path = (
    UPDATED_BASE_DIR
    / "figure5/functional_network_final-base_PER/"
      "geosmin_tarsal/tarsal_core_nodes.pkl"
)

thisG, dummy_nodes = add_dummy_nodes_from_pickle(
    thisG,
    pickle_path=dummy_node_path,
    color_info=color_info,
)


thisG = assign_node_modalities(
    thisG,
    tarsal_nodes=tarsal_sweetG.nodes,
    labial_nodes=labial_sweetG.nodes,
    dummy_nodes=dummy_nodes,
)


position_csv_path = (
    UPDATED_BASE_DIR
    / "figure5/functional_network_final-base_PER/"
      "merged_using_first_positions_0_1000.csv"
)

node_locations = load_node_locations(position_csv_path)

node_locations = add_missing_node_locations(
    thisG,
    node_locations,
    reference_node="horntail",
    x_spacing=100,
)

metaG_with_pos = set_node_location(
    thisG,
    node_locations,
)


metaG_with_pos_color = set_node_color(
    metaG_with_pos,
    color_info,
)


core_graph = threshold_and_transform_edges(
    metaG_with_pos_color,
    min_weight=7,
    weight_transform=np.log2,
)

core_graph = set_edge_color(core_graph)


nx.write_gexf(
    core_graph,
    "labial_av1a1_union.gexf",
)


core_graph_without_recurrent = remove_cross_modality_edges(
    core_graph,
    source_modality="labial",
    target_modality="geosmin",
)

core_graph_without_recurrent = set_edge_color(
    core_graph_without_recurrent
)


nx.write_gexf(
    core_graph_without_recurrent,
    "labial_av1a1_union_remove_recurrent.gexf",
)